In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('words.txt', 'r').read().splitlines()
words[:8]

['abakus',
 'abandon',
 'abazja',
 'abażur',
 'abażurek',
 'abdukcja',
 'abduktor',
 'abdykacja']

In [3]:
len(words)

48178

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'y', 24: 'z', 25: 'ó', 26: 'ą', 27: 'ć', 28: 'ę', 29: 'ł', 30: 'ń', 31: 'ś', 32: 'ź', 33: 'ż', 0: '.'}


In [5]:
# dataset

block_size = 3  # context size
X, Y = [], []

for w in words[:5]:
    print(w)
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '---->', itos[ix])
        context = context[1:] + [ix]  # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

abakus
... ----> a
..a ----> b
.ab ----> a
aba ----> k
bak ----> u
aku ----> s
kus ----> .
abandon
... ----> a
..a ----> b
.ab ----> a
aba ----> n
ban ----> d
and ----> o
ndo ----> n
don ----> .
abazja
... ----> a
..a ----> b
.ab ----> a
aba ----> z
baz ----> j
azj ----> a
zja ----> .
abażur
... ----> a
..a ----> b
.ab ----> a
aba ----> ż
baż ----> u
ażu ----> r
żur ----> .
abażurek
... ----> a
..a ----> b
.ab ----> a
aba ----> ż
baż ----> u
ażu ----> r
żur ----> e
ure ----> k
rek ----> .


In [12]:
X.shape

torch.Size([38, 3])

In [6]:
C = torch.randn((34, 2))

In [7]:
C

tensor([[-0.6706,  0.1306],
        [-1.2230, -0.0283],
        [ 2.4859, -0.3789],
        [ 0.0065, -0.6962],
        [ 0.1416,  1.4968],
        [-1.1387,  2.2108],
        [ 1.5752, -0.4567],
        [ 0.9627,  0.7612],
        [-0.4386,  0.9895],
        [ 0.6744, -0.6683],
        [-0.1333, -0.1844],
        [-0.2963, -0.3747],
        [-0.3286,  0.1621],
        [ 1.2470,  0.4743],
        [ 0.3112,  0.5762],
        [-1.0757,  0.7814],
        [-0.0681,  0.4902],
        [ 0.1926, -1.2369],
        [ 0.6803, -0.9110],
        [ 1.6113,  2.2200],
        [ 0.0909,  1.2532],
        [-1.1832,  1.3146],
        [-0.1648, -0.4636],
        [ 1.4779,  0.9622],
        [-1.3767, -1.8635],
        [-0.3213, -0.6508],
        [-0.2492, -0.9081],
        [-0.2220,  1.2436],
        [-0.0538, -1.6853],
        [ 0.7495, -1.5236],
        [-1.1276, -0.8955],
        [-0.1822, -0.3460],
        [-0.7164, -0.3478],
        [ 0.2832,  1.2795]])

In [10]:
C[5]

tensor([-1.1387,  2.2108])

In [8]:
F.one_hot(torch.tensor(5), num_classes=34)

tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
F.one_hot(torch.tensor(5), num_classes=34).float() @ C      # same as ^^ (all the zeros in F.one_hot... cancel values from C except for the 5th row)

tensor([-1.1387,  2.2108])

In [14]:
emb = C[X]
emb.shape

torch.Size([38, 3, 2])

In [15]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [ ]:
emb.shape, W1.shape     # dimensions don't fit, should be [38,6] and [6, 100]

(torch.Size([38, 3, 2]), torch.Size([6, 100]))

In [ ]:
emb = emb.view(-1, 6)   # now okay
emb.shape

torch.Size([38, 6])

In [18]:
h = emb @ W1 + b1
h.shape

torch.Size([38, 100])

In [19]:
W2 = torch.randn((100, 34))
b2 = torch.randn(34)

In [20]:
logits = h @ W2 + b2

In [21]:
logits.shape

torch.Size([38, 34])

In [22]:
counts = logits.exp()

In [23]:
probs = counts / counts.sum(1, keepdim=True)

In [24]:
probs.shape

torch.Size([38, 34])

In [27]:
probs[0].sum()      # sums to 1

tensor(1.)

In [ ]:
probs[torch.arange(38), Y]

tensor([2.3147e-15, 1.0261e-17, 4.2958e-37, 1.8260e-13, 1.1279e-05, 2.0563e-14,
        1.4977e-09, 2.3147e-15, 1.0261e-17, 4.2958e-37, 8.9724e-27, 1.7980e-16,
        0.0000e+00, 6.3333e-38, 1.8040e-09, 2.3147e-15, 1.0261e-17, 4.2958e-37,
        4.3047e-11, 0.0000e+00, 4.0716e-29, 4.8570e-24, 2.3147e-15, 1.0261e-17,
        4.2958e-37, 6.3058e-44, 7.4940e-13, 0.0000e+00, 3.7797e-17, 2.3147e-15,
        1.0261e-17, 4.2958e-37, 6.3058e-44, 7.4940e-13, 0.0000e+00, 4.5300e-19,
        1.4217e-02, 3.0646e-07])

In [ ]:
loss = -probs[torch.arange(38), Y].log().mean()     # inf is i thnk because some of the elements are zero
loss

tensor(inf)